# 01 — Data Ingestion: CSV → MongoDB

**Tujuan notebook ini (Fase 1.3 & 1.4 roadmap):**
- Load 9 CSV Olist dari `data/raw/`
- Insert ke collection masing-masing di MongoDB (`olist_db`), sebagai **raw storage layer**
- Validasi: CSV row count vs MongoDB document count harus match

> Prinsip Fase 0 #1: **raw data tidak langsung dimodifikasi**. Notebook ini cuma insert apa adanya, belum ada cleaning/transformasi sama sekali.


In [ ]:
import sys
!{sys.executable} -m pip install pymongo pandas

In [1]:
import pandas as pd
from pymongo import MongoClient
from pathlib import Path

# Ganti sesuai setup kamu (Atlas / lokal / Docker)
MONGO_URI = "mongodb://localhost:27017/"
DB_NAME = "olist_db"

RAW_DIR = Path("../data/raw")  # sesuaikan kalau struktur folder beda


### 1. Koneksi ke MongoDB

In [2]:
client = MongoClient(MONGO_URI)
db = client[DB_NAME]

# Test koneksi
print("Connected. Databases available:", client.list_database_names())


Connected. Databases available: ['admin', 'config', 'local']


### 2. Mapping CSV → Collection

Sesuai Fase 1.3 — struktur `olist_db`:
```
customers_raw, orders_raw, order_items_raw, payments_raw,
reviews_raw, products_raw, sellers_raw, geolocation_raw,
category_translation_raw
```


In [3]:
FILE_TO_COLLECTION = {
    "olist_customers_dataset.csv": "customers_raw",
    "olist_orders_dataset.csv": "orders_raw",
    "olist_order_items_dataset.csv": "order_items_raw",
    "olist_order_payments_dataset.csv": "payments_raw",
    "olist_order_reviews_dataset.csv": "reviews_raw",
    "olist_products_dataset.csv": "products_raw",
    "olist_sellers_dataset.csv": "sellers_raw",
    "olist_geolocation_dataset.csv": "geolocation_raw",
    "product_category_name_translation.csv": "category_translation_raw",
}


### 3. Ingestion Function

Insert apa adanya (tanpa cleaning), pakai `insert_many` untuk efisiensi.
Collection di-drop dulu kalau sudah ada isinya, supaya re-run notebook ini idempotent (tidak dobel data kalau dijalankan ulang).


In [4]:
def ingest_csv_to_mongo(csv_path: Path, collection_name: str):
    df = pd.read_csv(csv_path)
    records = df.to_dict(orient="records")

    collection = db[collection_name]
    collection.drop()  # supaya re-run tidak duplikat
    if records:
        collection.insert_many(records)

    return {
        "file": csv_path.name,
        "collection": collection_name,
        "csv_rows": len(df),
        "mongo_docs": collection.count_documents({}),
    }


### 4. Jalankan Ingestion untuk Semua File

In [5]:
results = []

for filename, collection_name in FILE_TO_COLLECTION.items():
    csv_path = RAW_DIR / filename
    if not csv_path.exists():
        print(f"⚠️  File tidak ditemukan: {csv_path}")
        continue

    result = ingest_csv_to_mongo(csv_path, collection_name)
    results.append(result)
    print(f"✅ {filename} → {collection_name}: {result['csv_rows']} rows → {result['mongo_docs']} docs")


✅ olist_customers_dataset.csv → customers_raw: 99441 rows → 99441 docs
✅ olist_orders_dataset.csv → orders_raw: 99441 rows → 99441 docs
✅ olist_order_items_dataset.csv → order_items_raw: 112650 rows → 112650 docs
✅ olist_order_payments_dataset.csv → payments_raw: 103886 rows → 103886 docs
✅ olist_order_reviews_dataset.csv → reviews_raw: 99224 rows → 99224 docs
✅ olist_products_dataset.csv → products_raw: 32951 rows → 32951 docs
✅ olist_sellers_dataset.csv → sellers_raw: 3095 rows → 3095 docs
✅ olist_geolocation_dataset.csv → geolocation_raw: 1000163 rows → 1000163 docs
✅ product_category_name_translation.csv → category_translation_raw: 71 rows → 71 docs


### 5. Validasi Ingestion (Fase 1.4)

Cek CSV row count vs MongoDB document count — **harus match**.


In [6]:
validation_df = pd.DataFrame(results)
validation_df["match"] = validation_df["csv_rows"] == validation_df["mongo_docs"]

print(validation_df.to_string(index=False))

if validation_df["match"].all():
    print("\n✅ Semua collection tervalidasi — row count sesuai.")
else:
    print("\n⚠️  Ada ketidaksesuaian row count, cek collection berikut:")
    print(validation_df[~validation_df["match"]])


                                 file               collection  csv_rows  mongo_docs  match
          olist_customers_dataset.csv            customers_raw     99441       99441   True
             olist_orders_dataset.csv               orders_raw     99441       99441   True
        olist_order_items_dataset.csv          order_items_raw    112650      112650   True
     olist_order_payments_dataset.csv             payments_raw    103886      103886   True
      olist_order_reviews_dataset.csv              reviews_raw     99224       99224   True
           olist_products_dataset.csv             products_raw     32951       32951   True
            olist_sellers_dataset.csv              sellers_raw      3095        3095   True
        olist_geolocation_dataset.csv          geolocation_raw   1000163     1000163   True
product_category_name_translation.csv category_translation_raw        71          71   True

✅ Semua collection tervalidasi — row count sesuai.


### 6. Cek Tambahan (opsional tapi disarankan)

- Missing `_id` (otomatis di-generate MongoDB, harusnya tidak ada masalah)
- Jumlah collection sesuai (9 collection)
- Field structure / data type tiap collection (bisa lanjut lebih detail di notebook 02 — Profiling)


In [7]:
print("Jumlah collection di database:", len(db.list_collection_names()))
print("Daftar collection:", db.list_collection_names())

# Contoh: intip struktur 1 dokumen dari orders_raw
sample_doc = db["orders_raw"].find_one()
print("\nContoh 1 dokumen dari orders_raw:")
print(sample_doc)


Jumlah collection di database: 9
Daftar collection: ['category_translation_raw', 'sellers_raw', 'geolocation_raw', 'orders_raw', 'payments_raw', 'products_raw', 'customers_raw', 'reviews_raw', 'order_items_raw']

Contoh 1 dokumen dari orders_raw:
{'_id': ObjectId('6a97b484b0c8756d6ac7d388'), 'order_id': 'e481f51cbdc54678b7cc49136f2d6af7', 'customer_id': '9ef432eb6251297304e76186b10a928d', 'order_status': 'delivered', 'order_purchase_timestamp': '2017-10-02 10:56:33', 'order_approved_at': '2017-10-02 11:07:15', 'order_delivered_carrier_date': '2017-10-04 19:55:00', 'order_delivered_customer_date': '2017-10-10 21:25:13', 'order_estimated_delivery_date': '2017-10-18 00:00:00'}


---

## Definition of Done (Fase 1)

- [ ] Semua 9 raw CSV masuk MongoDB
- [ ] Jumlah record sesuai (row count = document count)
- [ ] Struktur collection terdokumentasi (lanjut ke `docs/mongodb_schema.md`)
- [ ] Raw data tidak dimodifikasi (belum ada cleaning di notebook ini)

**Lanjut ke:** `02_data_profiling_validation.ipynb`
